# PO Delay Root Cause Analyzer: Data Pipeline and EDA

**Objetivo de la semana:** Cargar, limpiar, validar y explorar el dataset de Purchase Orders. Entregar un DataFrame limpio con todos los deltas calculados, listo para la clasificación de la semana 2.

**Output final:** df_clean.csv

---

### 1. Setup & Imports

In [ ]:
# Instalar dependencias
import os, warnings, matplotlib.pyplot as plt, matplotlib.ticker as mticker, numpy as np, pandas as pd, seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
from scipy import stats
#warnings.filterwarnings('ignore')

# Pipeline centralizado: una sola fuente de verdad para la limpieza y el cross-validation.
# Requiere que el kernel corra desde 01_data_pipeline_and_eda/ (donde vive pipeline_core.py).
from pipeline_core import clean_po_data, cross_validate_deltas

# ESTILO GLOBAL DE GRÁFICAS
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e8e8e8', 'grid.linewidth': 0.6, 'font.family': 'DejaVu Sans', 'axes.titlesize': 14, 'axes.titleweight': 'bold', 'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9, 'figure.dpi': 130})

# PALETAS DE COLORES CORPORATIVAS
PALETTE_MAIN, PALETTE_LATE, PALETTE_ONTIME, PALETTE_WARN = '#4C72B0', '#DD5C5C', '#55A868', '#F0A500'

print('✅ Setup completo')


## 2. Carga & Inventario de Columnas

In [ ]:

# DEFINICIÓN DE RUTA RAIZ
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "01_data_pipeline_and_eda":
    REPO_ROOT = REPO_ROOT.parent

load_dotenv(REPO_ROOT / ".env", override=True)

# RESOLVER RUTA AL CSV DESDE VARIABLE DE ENTORNO O USAR DEFAULT
_env_path = os.environ.get("PO_CSV_PATH") or None
CSV_PATH = Path(_env_path) if _env_path else REPO_ROOT / "data" / "raw" / "po_root_cause_synthetic.csv"

# VERIFICAR EXISTENCIA DEL CSV
if not CSV_PATH.exists():
    msg = (
        f"\n{'─'*60}\n"
        f"❌  CSV no encontrado\n\n"
        f"   Ruta buscada: {CSV_PATH}\n\n"
        f"   a) Coloca el archivo en data/raw/\n"
        f"      (ver data/raw/README.md)\n"
        f"   b) Define PO_CSV_PATH=/ruta/completa.csv en .env\n"
        f"{'─'*60}"
    )
    raise FileNotFoundError(msg)

# LEER CSV
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"✅ CSV cargado desde: {CSV_PATH}")
print(f"   Shape: {df_raw.shape}")

# 2. REPORTE DE CALIDAD (Diccionario en una sola línea)
null_report = pd.DataFrame({'dtype': df_raw.dtypes, 'nulls': df_raw.isnull().sum(), 'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(1), 'unique': df_raw.nunique()})
print('=== DIAGNÓSTICO DE COLUMNAS ===\n', null_report.to_string(), '\n\n=== PRIMERAS 5 FILAS DEL DATASET ===')

# 3. MUESTRA DE TABLA
display(df_raw.head(5))


## 3. Limpieza de Timestamps y Validación de Calidad

In [ ]:
# == LIMPIEZA Y ENRIQUECIMIENTO: UNA SOLA FUENTE DE VERDAD ==
# La lógica de limpieza vive en pipeline_core.clean_po_data() (mismo módulo que
# usa el script y que testea el CI). Aquí solo la invocamos; los prints de abajo
# son diagnóstico visible sobre el df ya enriquecido, no re-limpieza.
df = clean_po_data(df_raw)

DATE_COLS = ['PO_DT', 'STA_DT', 'RECPT_DT', 'REQUESTED_DT', 'FIRST_SUBMITTED_DT', 'DT_APPT_FIRST_APPROVED', 'APPROVED_DT', 'DT_APPT_CURRENT_APPROVED', 'PREVIOUS_REQUEST_DT', 'TRAILER_ARRIVE_DT', 'CHECKIN_DT', 'CHECKOUT_DT', 'TRAILER_DEPART_DT']

print('✅ Timestamps parseados. NaTs por columna:')
nat_counts = {c: df[c].isna().sum() for c in DATE_COLS if df[c].isna().sum() > 0}
for col, count in nat_counts.items(): print(f'   {col:<35} {count:>3} NaTs ({count/len(df)*100:.1f}%)')

# == DIAGNÓSTICO DE ORDEN CRONOLÓGICO (vista detallada) ==
# Nota: la columna _ts_issue que decide la confiabilidad la calcula clean_po_data();
# este desglose es informativo (incluye la tolerancia de 7d en TRAILER_ARRIVE vs APPROVED,
# que no entra en _ts_issue).
print('\n── Validación de orden cronológico ─────────────────────────────────────')
checks = [
    ('CHECKIN antes de TRAILER_ARRIVE',  df['CHECKIN_DT']        < df['TRAILER_ARRIVE_DT']),
    ('CHECKOUT antes de CHECKIN',         df['CHECKOUT_DT']       < df['CHECKIN_DT']),
    ('TRAILER_ARRIVE antes de APPROVED',  df['TRAILER_ARRIVE_DT'] < df['APPROVED_DT'] - pd.Timedelta(days=7)),
    ('RECPT antes de CHECKIN',            df['RECPT_DT']          < df['CHECKIN_DT']),
    ('STA antes de PO_DT',               df['STA_DT']            < df['PO_DT']),
]
for check_name, mask in checks:
    n = mask.sum()
    print(f"  {'⚠️ ' if n > 0 else '✅'} {check_name:<40} {n:>3} registros")

# Conteos de confiabilidad: directamente de las columnas que produjo clean_po_data()
print(f'\n  Total POs con algún problema de timestamp (_ts_issue): {df["_ts_issue"].sum()}\n  Total POs con TRAILER_ARRIVE_DT null:                  {df["_trailer_arrive_null"].sum()}\n  Total POs con datos confiables (_data_reliable):       {df["_data_reliable"].sum()}')

# == MÉTRICAS OPERACIONALES (ya calculadas por clean_po_data) ==
print(f'\n  POs con appointment reprogramado: {df["_rescheduled"].sum()} ({df["_rescheduled"].mean()*100:.1f}%)')
print(f'  Short ship POs (<90% fill rate):  {df["_short_ship"].sum()} ({df["_short_ship"].mean()*100:.1f}%)')


> **Decisión sobre los 39 POs no confiables.** De los 400 POs, 39 quedan
> marcados como no confiables: 12 con `_ts_issue` (timestamps fuera de orden,
> todos por CHECKOUT antes de CHECKIN) y 27 con `_trailer_arrive_null`
> (`TRAILER_ARRIVE_DT` nulo). **No se eliminan del dataset**: se conservan y se
> marcan con `_data_reliable = False` (calculado en `clean_po_data()` como
> `~_ts_issue & ~_trailer_arrive_null`). El flag queda disponible para que la
> **clasificación de Fase 2 los excluya o los trate con advertencia**; en Fase 1
> solo se etiquetan, no se filtran.


## 4. Cálculo de Deltas y Métricas

In [ ]:
# ── 4.1 Deltas de tiempo ─────────────────────────────────────────────────────
# Los deltas (lead_time_days, carrier_lag_hrs, yard_wait_calc_hrs, dock_calc_hrs,
# total_dc_hrs, appt_lead_days, delay_days_calc) ya los calculó clean_po_data().
# No se recalculan aquí: el df ya viene enriquecido.

# ── 4.2 Cross-validation: mis cálculos vs columnas pre-calculadas ────────────
# Centralizado en pipeline_core.cross_validate_deltas(): añade _yard_discrepancy /
# _dock_discrepancy e imprime el reporte (misma salida que antes, una sola fuente).
df = cross_validate_deltas(df)

# ── 4.3 Flags de clasificación preliminar (ya asignadas por clean_po_data) ────
print('\n── Flags de clasificación preliminar ───────────────────────────────────')
flags = ['flag_yard_congestion','flag_dock_backlog','flag_carrier_miss',
         'flag_short_lead_time','flag_hot_late']
for f in flags:
    n = df[f].sum()
    print(f'  {f:<30} {n:>3} POs ({n/len(df)*100:.1f}%)')

print('\n✅ Deltas calculados y flags asignadas')

## 5. EDA — Análisis exploratorio de Datos


### 5.1 Distribución Global de Delays

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribución Global de Purchase Orders', fontsize=15, fontweight='bold', y=1.01)

# ── Plot 1: On-Time vs Late ──────────────────────────────────────────────────
ax = axes[0]
late_counts = df['IS_LATE'].value_counts()
colors = [PALETTE_LATE if x == 'Y' else PALETTE_ONTIME for x in late_counts.index]
bars = ax.bar(['On-Time\n(N)', 'Late\n(Y)'],
              [late_counts.get('N', 0), late_counts.get('Y', 0)],
              color=[PALETTE_ONTIME, PALETTE_LATE], width=0.5, edgecolor='white')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 3, f'{h}\n({h/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('On-Time vs Late')
ax.set_ylabel('Número de POs')
ax.set_ylim(0, 280)

# ── Plot 2: Distribución de DELAY_DAYS (solo tardíos) ───────────────────────
ax = axes[1]
late_df = df[df['IS_LATE'] == 'Y']
ax.hist(late_df['DELAY_DAYS'], bins=18, color=PALETTE_LATE, alpha=0.85, edgecolor='white')
ax.axvline(late_df['DELAY_DAYS'].mean(), color='#222', linestyle='--', linewidth=1.5,
           label=f'Media: {late_df["DELAY_DAYS"].mean():.2f}d')
ax.axvline(late_df['DELAY_DAYS'].median(), color='#888', linestyle=':', linewidth=1.5,
           label=f'Mediana: {late_df["DELAY_DAYS"].median():.2f}d')
ax.legend()
ax.set_title('Días de Delay (solo POs tardíos)')
ax.set_xlabel('Días de retraso')
ax.set_ylabel('Frecuencia')

# ── Plot 3: Reason Code distribution ────────────────────────────────────────
ax = axes[2]
reason_counts = df['REASON_DSC'].value_counts().drop('Not applicable', errors='ignore')
reason_counts = reason_counts.sort_values(ascending=True)
colors_reason = [PALETTE_WARN if 'Rescheduled' in r or 'Vendor' in r
                 else PALETTE_LATE if 'Carrier' in r or 'Missed' in r or 'Equipment' in r
                 else PALETTE_MAIN for r in reason_counts.index]
bars = ax.barh(reason_counts.index, reason_counts.values,
               color=colors_reason, edgecolor='white', height=0.6)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height()/2, str(int(w)),
            va='center', fontsize=9)
ax.set_title('Reason Codes (POs tardíos)')
ax.set_xlabel('Número de POs')

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=PALETTE_WARN,   label='Vendor/Scheduling'),
    Patch(facecolor=PALETTE_LATE,   label='Carrier/External'),
    Patch(facecolor=PALETTE_MAIN,   label='DC Operations'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('eda_01_distribucion_global.png', dpi=130, bbox_inches='tight')
plt.show()

# Estadísticas descriptivas
print('── Estadísticas de DELAY_DAYS (todos los POs) ──────────────────────────')
print(df['DELAY_DAYS'].describe().round(3))
print(f'\nDelay medio (solo tardíos): {late_df["DELAY_DAYS"].mean():.2f} días')
print(f'POs con >3 días de retraso: {(df["DELAY_DAYS"]>3).sum()}')

### 5.2 Análisis por Distribution Center

### 5.3 Análisis por Vendor y Carrier

### 5.4 Reason Code Analysis + Mismatch Detection

### 5.5 HOT POs, Short Shipments y Casos Críticos

### 5.6 Mapa de Correlaciones

## 6. Verificación del pipeline centralizado (`pipeline_core`)

La limpieza (`clean_po_data`) y el cross-validation (`cross_validate_deltas`) viven en
`pipeline_core.py` — la misma fuente que usa el script y que cubre el CI. Esta sección
confirma que el import produce las mismas cifras (no duplica la lógica).

In [ ]:
# ── Verificación: el módulo produce las mismas cifras ────────────────────────
# clean_po_data() y cross_validate_deltas() viven en pipeline_core.py (importadas
# en la celda de setup). Aquí confirmamos que el import produce el mismo df_clean
# que el resto del notebook usa, como verificación visible de paridad.
df_clean = clean_po_data(df_raw)
df_clean = cross_validate_deltas(df_clean)

print('✅ clean_po_data() ejecutado correctamente')
print(f'   Shape:                      {df_clean.shape}')
print(f'   Columnas agregadas:         {df_clean.shape[1] - df_raw.shape[1]}')
print(f'   POs con datos confiables:   {df_clean["_data_reliable"].sum()} / {len(df_clean)}')
print(f'   POs rescheduled:            {df_clean["_rescheduled"].sum()}')
print(f'   Short ships:                {df_clean["_short_ship"].sum()}')
print(f'   Flag yard congestion:       {df_clean["flag_yard_congestion"].sum()}')
print(f'   Flag dock backlog:          {df_clean["flag_dock_backlog"].sum()}')
print(f'   Flag carrier miss:          {df_clean["flag_carrier_miss"].sum()}')

# Nuevas columnas
new_cols = [c for c in df_clean.columns if c not in df_raw.columns]
print(f'\nColumnas nuevas: {new_cols}')

## 7. Guardar Output y Resumen de Hallazgos